In [ ]:
SMALL = False

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_colwidth", 120)
DRIVE_ROOT = Path("/content/drive/MyDrive/llm-persona-ei")

def find(name, candidates):
    path = next((path for path in candidates if path.exists()), None)
    if path is None and IN_COLAB:
        print(f"{name} not found: please upload it")
        path = Path(next(iter(files.upload())))
    assert path is not None, f"{name} not found in {[str(c) for c in candidates]}"
    return path


CONFIG_PATH = find("config.yaml", [Path("config.yaml"), Path("../config.yaml"), DRIVE_ROOT / "config.yaml"])
ROOT = Path(os.environ.get("PERSONA_EI_RESULTS") or (DRIVE_ROOT if IN_COLAB else CONFIG_PATH.resolve().parent))

OUT_DIR = ROOT / "comparison" / "results"
OUT_DIR = OUT_DIR / "small" if SMALL else OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("config   :", CONFIG_PATH.resolve())
print("results  :", ROOT)
print("writes to:", OUT_DIR)

In [ ]:
cfg = yaml.safe_load(CONFIG_PATH.read_text())
QUESTIONS = {"rq1_persona": "RQ1 persona consistency", "rq2_emotion": "RQ2 emotion recognition",
             "rq3_sycophancy": "RQ3 sycophancy", "rq4_decisions": "RQ4 decisions"}


def tables_dir(folder):
    results = ROOT / folder / "results"
    return (results / "small" if SMALL else results) / "tables"


def read(name):
    frames = []
    for folder, question in QUESTIONS.items():
        path = tables_dir(folder) / f"{name}.csv"
        frame = pd.read_csv(path) if path.exists() and path.stat().st_size else pd.DataFrame()
        if not frame.empty:
            frames.append(frame.assign(question=question))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def main_measure(table):
    return table[table["measure"] == table.groupby("question")["measure"].transform("first")]


def side_by_side(table, by, order):
    main = main_measure(table)
    interval = [f"{row.mean:.3f}" + (f" [{row.lo:.3f}, {row.hi:.3f}]" if pd.notna(row.lo) else "") for row in main.itertuples()]
    wide = main.assign(column=main["question"] + ": " + main["measure"], value=interval)
    wide = wide.pivot(index=by, columns="column", values="value")
    return wide.reindex([name for name in order if name in wide.index]).fillna("").reset_index()


for folder, question in QUESTIONS.items():
    print(f"{question:26s}", "results found" if (tables_dir(folder) / "by_model.csv").exists() else "no results yet")
by_model, by_persona = read("by_model"), read("by_persona")
assert not by_model.empty, "no results yet: run at least one question notebook first"

tables = {}
main = read("primary_result")
counts = [column for column in ("replies", "items", "conversations", "cells") if column in main]
main["n"] = main[counts].bfill(axis=1).iloc[:, 0]
tables["main_results"] = main.reindex(columns=["question", "result", "model", "mean", "lo", "hi", "p_value", "n"])

specs = pd.DataFrame(cfg["models"]).T[["family", "params_b"]].rename_axis("model").reset_index()
by_model = by_model.merge(specs, on="model", how="left")
tables["by_model"] = by_model[["question", "measure", "model", "family", "params_b", "mean", "lo", "hi", "n"]]
models = [*[key for key in cfg["models"] if key in set(by_model["model"])],
          *[key for key in dict.fromkeys(by_model["model"]) if key not in cfg["models"]]]
tables["model_comparison"] = specs.merge(side_by_side(by_model, "model", models), on="model", how="right")
tables["coverage"] = main_measure(by_model).pivot(index="model", columns="question", values="n").reindex(models).reset_index()
if not by_persona.empty:
    tables["by_persona"] = by_persona[["question", "measure", "persona", "mean", "lo", "hi", "n"]]
    personas = list(dict.fromkeys(["neutral", *by_persona["persona"]]))
    tables["persona_comparison"] = side_by_side(by_persona, "persona", personas)

print("\nMAIN RESULTS\n" + tables["main_results"].round(4).to_string(index=False))
print("\nMODEL COMPARISON (main measure of each question, with its 95% interval)\n"
      + tables["model_comparison"].to_string(index=False, na_rep=""))

In [ ]:
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TEXT, MUTED, GRID, SURFACE = "#52514e", "#898781", "#e1e0d9", "#fcfcfb"


def chart(table, by, order, title, reference=()):
    questions = list(dict.fromkeys(table["question"]))
    figure, axes = plt.subplots(1, len(questions), figsize=(3.7 * len(questions), 3.9), squeeze=False, facecolor=SURFACE)
    for axis, question in zip(axes.flat, questions):
        part = table[table["question"] == question].set_index(by)
        bars = part.reindex([name for name in order if name in part.index])
        places = np.arange(len(bars))
        errors = np.nan_to_num(np.vstack([bars["mean"] - bars["lo"], bars["hi"] - bars["mean"]]).astype(float)).clip(min=0)
        tops = (bars["mean"] + errors[1]).to_numpy(dtype=float)
        axis.bar(places, bars["mean"], width=0.4, color=SERIES[0], yerr=errors, ecolor=TEXT, capsize=3, error_kw={"elinewidth": 1})
        for place, value, top in zip(places, bars["mean"], tops):
            label = f"{value:.3f}" if 0 < abs(value) < 0.1 else f"{value:.2f}"
            axis.annotate(label, (place, max(top, 0)), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8, color=TEXT)
        lines = part[part.index.isin(reference)]
        for name, row in lines.iterrows():
            axis.axhline(row["mean"], color=MUTED, linewidth=1)
            axis.annotate(name, (1, row["mean"]), xycoords=("axes fraction", "data"), xytext=(-2, 3),
                          textcoords="offset points", ha="right", fontsize=8, color=TEXT)
        highest = np.nanmax([0, *tops, *lines["mean"]])
        lowest = np.nanmin([0, *(bars["mean"] - errors[0])])
        axis.set_ylim(1.2 * lowest, 1.2 * highest if highest > 0 else 1)
        axis.set_xticks(places)
        axis.set_xticklabels(bars.index, rotation=30, ha="right")
        axis.set_xlim(-0.7, max(len(bars), 4) - 0.3)
        axis.set_title(question, fontsize=10, loc="left")
        axis.set_ylabel(part["measure"].iloc[0], fontsize=9)
        axis.set_facecolor(SURFACE)
        axis.grid(axis="y", color=GRID, linewidth=1)
        axis.set_axisbelow(True)
        axis.tick_params(colors=MUTED, labelsize=8, length=0)
        axis.xaxis.label.set_color(TEXT)
        axis.yaxis.label.set_color(TEXT)
        for side in ("top", "right", "left"):
            axis.spines[side].set_visible(False)
        axis.spines["bottom"].set_color(GRID)
    figure.suptitle(title, x=0.01, ha="left", fontsize=11)
    figure.tight_layout()
    return figure


figures = {}
tested = [key for key in models if key in cfg["models"]]
others = [key for key in models if key not in cfg["models"]]
views = {"models": ("All models", tested),
         "models_by_size": ("One family at four sizes", cfg["groups"]["size"]),
         "models_by_family": ("Four makers at one size", cfg["groups"]["family"])}
for name, (title, members) in views.items():
    members = [key for key in members if key in tested]
    if members:
        figures[name] = chart(main_measure(by_model), "model", members, title, reference=others)
    else:
        print(f"{name}: none of these models has results yet")
if not by_persona.empty:
    figures["personas"] = chart(main_measure(by_persona), "persona", personas, "The personas, over all models")

In [ ]:
for name in ("persona_comparison", "coverage"):
    if name in tables:
        print(f"{name.upper()}\n" + tables[name].to_string(index=False, na_rep="") + "\n")

In [ ]:
(OUT_DIR / "tables").mkdir(exist_ok=True)
(OUT_DIR / "figures").mkdir(exist_ok=True)
for name, content in tables.items():
    content.to_csv(OUT_DIR / "tables" / f"{name}.csv", index=False)
for name, figure in figures.items():
    figure.savefig(OUT_DIR / "figures" / f"{name}.png", dpi=130, bbox_inches="tight")
(OUT_DIR / "run_info.json").write_text(json.dumps({
    "small": SMALL, "questions": sorted(set(by_model["question"])), "models": models,
}, indent=2))
print("saved to", OUT_DIR)
print("tables:", ", ".join(tables))
print("figures:", ", ".join(figures))